# 1. Data Prep — Build the Master Table

Loads all 9 Olist CSVs, joins them into one order-item-level table, and
saves it for the analysis notebooks. 

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 120)

RAW_DIR = Path("../../")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, PROCESSED_DIR

(WindowsPath('../..'), WindowsPath('../data/processed'))

In [2]:
print(RAW_DIR)

..\..


## Load raw tables

In [3]:
orders = pd.read_csv(
    RAW_DIR / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv", parse_dates=["shipping_limit_date"])
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv",
                      parse_dates=["review_creation_date", "review_answer_timestamp"])
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
category_translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

print({name: df.shape for name, df in [
    ("orders", orders), ("order_items", order_items), ("payments", payments),
    ("reviews", reviews), ("products", products), ("sellers", sellers),
    ("customers", customers), ("category_translation", category_translation),
]})

{'orders': (99441, 8), 'order_items': (112650, 7), 'payments': (103886, 5), 'reviews': (99224, 7), 'products': (32951, 9), 'sellers': (3095, 4), 'customers': (99441, 5), 'category_translation': (71, 2)}


## Collapse the one-to-many tables to one row per order

An order can have multiple payments (split across methods) and, rarely,
more than one review. We aggregate those down to a single row per
`order_id` *before* merging, so we don't accidentally duplicate order rows.

In [4]:
payments_agg = (
    payments.groupby("order_id")
    .agg(
        payment_value_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_primary=("payment_type", lambda s: s.value_counts().idxmax()),
        n_payment_methods=("payment_type", "nunique"),
    )
    .reset_index()
)

reviews_dedup = (
    reviews.sort_values("review_answer_timestamp")
    .drop_duplicates(subset="order_id", keep="last")[
        ["order_id", "review_score", "review_comment_title", "review_comment_message"]
    ]
)

products = products.merge(category_translation, on="product_category_name", how="left")
products["product_category_name_english"] = products["product_category_name_english"].fillna("unknown")

payments_agg.shape, reviews_dedup.shape

((99440, 5), (98673, 4))

## Build the master table (grain: one row per order line-item)

In [5]:
master = (
    order_items
    .merge(orders, on="order_id", how="left")
    .merge(products[["product_id", "product_category_name_english", "product_weight_g",
                      "product_length_cm", "product_height_cm", "product_width_cm"]],
           on="product_id", how="left")
    .merge(sellers, on="seller_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_dedup, on="order_id", how="left")
)

master["item_total_value"] = master["price"] + master["freight_value"]

delivered_mask = master["order_delivered_customer_date"].notna()
master.loc[delivered_mask, "delivery_days"] = (
    master.loc[delivered_mask, "order_delivered_customer_date"]
    - master.loc[delivered_mask, "order_purchase_timestamp"]
).dt.days
master.loc[delivered_mask, "was_late"] = (
    master.loc[delivered_mask, "order_delivered_customer_date"]
    > master.loc[delivered_mask, "order_estimated_delivery_date"]
)

master["order_purchase_month"] = master["order_purchase_timestamp"].dt.to_period("M").astype(str)

print("master shape:", master.shape)
master.head()

master shape: (112650, 37)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,payment_installments_max,payment_type_primary,n_payment_methods,review_score,review_comment_title,review_comment_message,item_total_value,delivery_days,was_late,order_purchase_month
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,2.0,credit_card,1.0,5.0,NaN,"Perfeito, produto entregue antes do combinado.",72.19,7.0,False,2017-09
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,3.0,credit_card,1.0,4.0,NaN,NaN,259.83,16.0,False,2017-04
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,5.0,credit_card,1.0,5.0,NaN,Chegou antes do prazo previsto e o produto sur...,216.87,7.0,False,2018-01
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,2.0,credit_card,1.0,4.0,NaN,NaN,25.78,6.0,False,2018-08
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,3.0,credit_card,1.0,5.0,NaN,Gostei pois veio no prazo determinado .,218.04,25.0,False,2017-02


## Sanity check

A handful of orders (mostly `unavailable` / `canceled` status) have no
line items at all, so `master` will have slightly fewer unique orders
than the raw `orders` table. That's expected — not a bug.

In [6]:
print("orders in master:", master["order_id"].nunique())
print("orders in raw table:", orders["order_id"].nunique())
print("missing from master:", orders["order_id"].nunique() - master["order_id"].nunique())

orders in master: 98666
orders in raw table: 99441
missing from master: 775


## Save for the analysis notebooks

In [7]:
master.to_csv(PROCESSED_DIR / "master_order_items.csv", index=False)
print("saved:", PROCESSED_DIR / "master_order_items.csv")

saved: ..\data\processed\master_order_items.csv
